# Multiclass Classification Pipeline for DCombo

This notebook trains and evaluates multiple classification models for predicting DCombo using scikit-learn pipelines with stratified cross-validation.

## Requirements
- Python 3.8+
- scikit-learn
- pandas
- numpy
- xgboost (optional)

## Usage
1. Ensure `X_train.csv` and `Y_train.csv` are in the same directory as this notebook
2. Run all cells in sequence
3. The notebook will generate the following output files:
   - `labels_mapping.csv` - Mapping of class labels to indices
   - `cv_results_<model>.csv` - Cross-validation metrics for each model
   - `confusion_matrix.csv` - Confusion matrix for the best model
   - `classification_report.csv` - Classification report for the best model
   - `feature_importances.csv` - Feature importances (if applicable)

## Models
The script trains and compares the following models:
1. **RandomForestClassifier** - Ensemble of decision trees
2. **XGBClassifier** - Gradient boosting (if xgboost is installed)
3. **MultinomialNB** - Naive Bayes classifier

The best model is selected based on F1-macro score.

## Key Features
- Automatic detection of numeric and categorical features
- Proper preprocessing pipeline with ColumnTransformer
- Stratified 6-fold cross-validation
- Hyperparameter tuning with RandomizedSearchCV
- Comprehensive evaluation metrics (accuracy, F1-macro, precision, recall)
- No data leakage - all preprocessing inside pipelines

## 1. Imports and Setup

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    StratifiedKFold, cross_validate, cross_val_predict, RandomizedSearchCV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    confusion_matrix, classification_report, make_scorer,
    accuracy_score, f1_score, precision_score, recall_score
)

# Try to import XGBoost
XGBOOST_AVAILABLE = False
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("✓ XGBoost is available")
except ImportError:
    print("✗ XGBoost not available, will skip XGBClassifier")

# Set random state for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

✓ XGBoost is available


## 2. Load Data

In [2]:
# Load training data
print("Loading data...")
X_train = pd.read_csv('X_train.csv')
Y_train_df = pd.read_csv('Y_train.csv')

# Nota: La limpieza de columnas (fecha, NaN >20%) se hará en build_preprocessor()
y_train = Y_train_df['DCombo'].astype(str)  # Ensure categorical/string type

print(f"✓ Data loaded: {X_train.shape[0]} samples, {X_train.shape[1]} features (antes de limpieza)")
print(f"✓ Target classes: {sorted(y_train.unique())}")
print(f"\nClass distribution:")
print(y_train.value_counts().sort_index())

Loading data...
✓ Data loaded: 132 samples, 49 features (antes de limpieza)
✓ Target classes: ['C', 'M1', 'M2', 'MM']

Class distribution:
DCombo
C     44
M1    39
M2    31
MM    18
Name: count, dtype: int64


## 3. Helper Functions

In [3]:
def build_preprocessor(X, nan_threshold=0.20):
    """
    Build a ColumnTransformer for preprocessing numeric and categorical features.
    Removes columns with >20% NaN and 'fecha' column.
    
    Parameters:
    -----------
    X : DataFrame or array-like
        Training features
    nan_threshold : float, default=0.20
        Maximum proportion of NaN values allowed (columns above this are dropped)
    
    Returns:
    --------
    tuple: (ColumnTransformer, DataFrame)
        Preprocessor and cleaned DataFrame
    """
    if isinstance(X, pd.DataFrame):
        X_clean = X.copy()
        
        # 1. Eliminar columna 'fecha' si existe
        if 'fecha' in X_clean.columns:
            print(f"  → Eliminando columna 'fecha' (no aporta información predictiva)")
            X_clean = X_clean.drop(columns=['fecha'])
        
        # 2. Eliminar columnas con más del threshold% de NaN
        nan_percentage = X_clean.isna().sum() / len(X_clean)
        cols_to_drop = nan_percentage[nan_percentage > nan_threshold].index.tolist()
        
        if cols_to_drop:
            print(f"  → Eliminando {len(cols_to_drop)} columnas con >{nan_threshold*100:.0f}% NaN:")
            for col in cols_to_drop:
                print(f"      • {col}: {nan_percentage[col]*100:.1f}% NaN")
            X_clean = X_clean.drop(columns=cols_to_drop)
        
        # 3. Detect numeric and categorical columns
        numeric_features = X_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
        categorical_features = X_clean.select_dtypes(include=['object', 'category']).columns.tolist()
    else:
        # If not DataFrame, treat all as numeric
        X_clean = X
        n_features = X.shape[1] if hasattr(X, 'shape') else len(X[0])
        numeric_features = [f"feature_{i}" for i in range(n_features)]
        categorical_features = []
    
    print(f"  Features finales: {X_clean.shape[1]} columnas")
    print(f"    - Numéricas: {len(numeric_features)}")
    print(f"    - Categóricas: {len(categorical_features)}")
    
    # Build transformers
    transformers = []
    
    if numeric_features:
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler(with_mean=False))  # with_mean=False for OHE compatibility
        ])
        transformers.append(('num', numeric_transformer, numeric_features))
    
    if categorical_features:
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
        ])
        transformers.append(('cat', categorical_transformer, categorical_features))
    
    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder='drop'
    )
    
    return preprocessor, X_clean

In [5]:
def build_models(preprocessor):
    """
    Build dictionary of models with their pipelines and hyperparameter grids.
    
    Parameters:
    -----------
    preprocessor : ColumnTransformer
        Fitted preprocessor
    
    Returns:
    --------
    dict
        Dictionary with model names as keys and (pipeline, param_grid, needs_label_encoding) tuples as values
    """
    models = {}
    
    # Random Forest
    rf_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=RANDOM_STATE))
    ])
    rf_param_grid = {
        'classifier__n_estimators': [50, 100, 200, 300],
        'classifier__max_depth': [None, 5, 10, 15, 20],
        'classifier__min_samples_split': [2, 5, 10],
        'classifier__min_samples_leaf': [1, 2, 4],
        'classifier__max_features': ['sqrt', 'log2', None]
    }
    models['random_forest'] = (rf_pipeline, rf_param_grid, False)  # No label encoding needed
    
    # XGBoost (if available) - needs label encoding
    if XGBOOST_AVAILABLE:
        xgb_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', XGBClassifier(random_state=RANDOM_STATE, eval_metric='mlogloss'))
        ])
        xgb_param_grid = {
            'classifier__n_estimators': [50, 100, 200, 300],
            'classifier__max_depth': [3, 5, 7, 9],
            'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'classifier__subsample': [0.6, 0.8, 1.0],
            'classifier__colsample_bytree': [0.6, 0.8, 1.0]
        }
        models['xgb'] = (xgb_pipeline, xgb_param_grid, True)  # Needs label encoding
    
    # Multinomial Naive Bayes
    # Note: MultinomialNB requires non-negative features
    # We'll use it with the preprocessor which outputs sparse matrix from OHE
    nb_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', MultinomialNB())
    ])
    nb_param_grid = {
        'classifier__alpha': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
    }
    models['nb'] = (nb_pipeline, nb_param_grid, False)  # No label encoding needed
    
    return models

In [6]:
def evaluate_with_cv(model, X, y, cv, scoring):
    """
    Evaluate model using cross-validation.
    
    Parameters:
    -----------
    model : Pipeline
        Model pipeline to evaluate
    X : DataFrame or array-like
        Features
    y : Series or array-like
        Target variable
    cv : cross-validation generator
        Cross-validation strategy
    scoring : dict
        Scoring metrics
    
    Returns:
    --------
    dict
        Cross-validation results
    """
    cv_results = cross_validate(
        model, X, y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )
    return cv_results

In [7]:
def fit_and_report(model_name, pipeline, param_grid, X, y, cv, scoring, needs_label_encoding=False):
    """
    Fit model with hyperparameter tuning and generate reports.
    
    Parameters:
    -----------
    model_name : str
        Name of the model
    pipeline : Pipeline
        Model pipeline
    param_grid : dict
        Hyperparameter grid
    X : DataFrame or array-like
        Features
    y : Series or array-like
        Target variable
    cv : cross-validation generator
        Cross-validation strategy
    scoring : dict
        Scoring metrics
    needs_label_encoding : bool
        Whether to encode labels to integers
    
    Returns:
    --------
    tuple
        (best_pipeline, cv_results_df, mean_f1_macro, label_encoder)
    """
    print(f"\n{'='*60}")
    print(f"Training {model_name}...")
    print(f"{'='*60}")
    
    # Handle label encoding if needed
    label_encoder = None
    y_encoded = y
    if needs_label_encoding:
        print("Encoding labels for XGBoost...")
        label_encoder = LabelEncoder()
        y_encoded = label_encoder.fit_transform(y)
    
    # Hyperparameter tuning with RandomizedSearchCV
    print("Performing hyperparameter search...")
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid,
        n_iter=20,
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        error_score='raise'
    )
    
    try:
        search.fit(X, y_encoded)
        best_pipeline = search.best_estimator_
        
        print(f"✓ Best parameters: {search.best_params_}")
        print(f"✓ Best f1_macro score: {search.best_score_:.4f}")
        
        # Evaluate best model with cross_validate to get detailed metrics
        print("Evaluating best model with cross-validation...")
        cv_results = evaluate_with_cv(best_pipeline, X, y_encoded, cv, scoring)
        
        # Create results DataFrame
        results_data = {
            'fold': list(range(1, len(cv_results['test_accuracy']) + 1)),
            'accuracy': cv_results['test_accuracy'],
            'f1_macro': cv_results['test_f1_macro'],
            'precision_macro': cv_results['test_precision_macro'],
            'recall_macro': cv_results['test_recall_macro']
        }
        cv_results_df = pd.DataFrame(results_data)
        
        # Print summary
        print("\nCross-validation results:")
        print(cv_results_df)
        print("\nMean scores:")
        print(f"  Accuracy:  {cv_results_df['accuracy'].mean():.4f} ± {cv_results_df['accuracy'].std():.4f}")
        print(f"  F1-macro:  {cv_results_df['f1_macro'].mean():.4f} ± {cv_results_df['f1_macro'].std():.4f}")
        print(f"  Precision: {cv_results_df['precision_macro'].mean():.4f} ± {cv_results_df['precision_macro'].std():.4f}")
        print(f"  Recall:    {cv_results_df['recall_macro'].mean():.4f} ± {cv_results_df['recall_macro'].std():.4f}")
        
        # Save results
        output_file = f"cv_results_{model_name}.csv"
        cv_results_df.to_csv(output_file, index=False)
        print(f"\n✓ Results saved to {output_file}")
        
        mean_f1 = cv_results_df['f1_macro'].mean()
        return best_pipeline, cv_results_df, mean_f1, label_encoder
        
    except Exception as e:
        print(f"✗ Error training {model_name}: {str(e)}")
        return None, None, 0.0, None

In [8]:
def create_labels_mapping(y):
    """
    Create and save labels mapping table.
    
    Parameters:
    -----------
    y : Series or array-like
        Target variable
    """
    # Get unique classes and their counts
    classes = np.unique(y)
    value_counts = pd.Series(y).value_counts()
    
    # Create mapping DataFrame
    labels_df = pd.DataFrame({
        'label_index': range(len(classes)),
        'label_name': classes,
        'support': [value_counts[c] for c in classes]
    })
    
    # Save to CSV
    labels_df.to_csv('labels_mapping.csv', index=False)
    print("\nLabels mapping:")
    print(labels_df)
    print("\n✓ Labels mapping saved to labels_mapping.csv")

In [9]:
def generate_best_model_reports(best_pipeline, model_name, X, y, cv, label_encoder=None):
    """
    Generate confusion matrix, classification report, and feature importances for best model.
    
    Parameters:
    -----------
    best_pipeline : Pipeline
        Best trained pipeline
    model_name : str
        Name of the best model
    X : DataFrame or array-like
        Features
    y : Series or array-like
        Target variable
    cv : cross-validation generator
        Cross-validation strategy
    label_encoder : LabelEncoder, optional
        Label encoder if used for this model
    """
    print(f"\n{'='*60}")
    print(f"Generating reports for best model: {model_name}")
    print(f"{'='*60}")
    
    # Encode labels if needed
    y_for_model = y
    if label_encoder is not None:
        y_for_model = label_encoder.transform(y)
    
    # Get cross-validated predictions
    print("Generating cross-validated predictions...")
    y_pred = cross_val_predict(best_pipeline, X, y_for_model, cv=cv, method='predict')
    
    # Decode predictions if needed
    if label_encoder is not None:
        y_pred = label_encoder.inverse_transform(y_pred)
    
    # Confusion Matrix
    print("\nCreating confusion matrix...")
    cm = confusion_matrix(y, y_pred)
    classes = sorted(np.unique(y))
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)
    cm_df.to_csv('confusion_matrix.csv')
    print("Confusion Matrix:")
    print(cm_df)
    print("✓ Confusion matrix saved to confusion_matrix.csv")
    
    # Classification Report
    print("\nCreating classification report...")
    report_dict = classification_report(y, y_pred, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv('classification_report.csv')
    print("Classification Report:")
    print(report_df)
    print("✓ Classification report saved to classification_report.csv")
    
    # Feature Importances (if available)
    print("\nChecking for feature importances...")
    classifier = best_pipeline.named_steps['classifier']
    
    if hasattr(classifier, 'feature_importances_'):
        print("Extracting feature importances...")
        
        # Get feature names from preprocessor
        preprocessor = best_pipeline.named_steps['preprocessor']
        feature_names = preprocessor.get_feature_names_out()
        
        # Create importances DataFrame
        importances_df = pd.DataFrame({
            'feature': feature_names,
            'importance': classifier.feature_importances_
        })
        importances_df = importances_df.sort_values('importance', ascending=False)
        importances_df.to_csv('feature_importances.csv', index=False)
        
        print("\nTop 20 Feature Importances:")
        print(importances_df.head(20))
        print("✓ Feature importances saved to feature_importances.csv")
    elif hasattr(classifier, 'coef_'):
        print("Model has coefficients but not feature importances.")
        print("(Skipping feature importance export for this model type)")
    else:
        print("Model does not support feature importances.")

## 4. Build Preprocessor

In [10]:
print("\nBuilding preprocessor...")
preprocessor, X_train = build_preprocessor(X_train, nan_threshold=0.20)
print("✓ Preprocessor built successfully")
print(f"✓ Features después de limpieza: {X_train.shape[1]} columnas")


Building preprocessor...
  → Eliminando columna 'fecha' (no aporta información predictiva)
  → Eliminando 2 columnas con >20% NaN:
      • origen antepasados (extranjeros): 74.2% NaN
      • hº act.cerca sem: 100.0% NaN
  Features finales: 46 columnas
    - Numéricas: 32
    - Categóricas: 14
✓ Preprocessor built successfully
✓ Features después de limpieza: 46 columnas


## 5. Define Cross-Validation Strategy and Scoring Metrics

In [14]:
# Stratified K-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"Cross-validation: {cv.n_splits}-fold stratified")

# Scoring metrics
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'f1_macro': make_scorer(f1_score, average='macro'),
    'precision_macro': make_scorer(precision_score, average='macro', zero_division=0),
    'recall_macro': make_scorer(recall_score, average='macro', zero_division=0)
}
print("Metrics: accuracy, f1_macro, precision_macro, recall_macro")

Cross-validation: 5-fold stratified
Metrics: accuracy, f1_macro, precision_macro, recall_macro


## 6. Build and Train Models

In [15]:
print("\nBuilding models...")
models = build_models(preprocessor)
print(f"✓ {len(models)} models prepared: {list(models.keys())}")


Building models...
✓ 3 models prepared: ['random_forest', 'xgb', 'nb']


In [16]:
# Train and evaluate all models
results = {}

for model_name, (pipeline, param_grid, needs_encoding) in models.items():
    best_pipe, cv_results_df, mean_f1, label_enc = fit_and_report(
        model_name, pipeline, param_grid, X_train, y_train, cv, scoring, needs_encoding
    )
    if best_pipe is not None:
        results[model_name] = {
            'pipeline': best_pipe,
            'cv_results': cv_results_df,
            'mean_f1_macro': mean_f1,
            'label_encoder': label_enc
        }


Training random_forest...
Performing hyperparameter search...
✓ Best parameters: {'classifier__n_estimators': 100, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 2, 'classifier__max_features': None, 'classifier__max_depth': 20}
✓ Best f1_macro score: 0.6591
Evaluating best model with cross-validation...

Cross-validation results:
   fold  accuracy  f1_macro  precision_macro  recall_macro
0     1  0.666667  0.696429         0.729167      0.684028
1     2  0.666667  0.666667         0.707418      0.678571
2     3  0.730769  0.741342         0.737302      0.753968
3     4  0.461538  0.542076         0.540476      0.548611
4     5  0.615385  0.649107         0.731456      0.618056

Mean scores:
  Accuracy:  0.6282 ± 0.1018
  F1-macro:  0.6591 ± 0.0742
  Precision: 0.6892 ± 0.0839
  Recall:    0.6566 ± 0.0772

✓ Results saved to cv_results_random_forest.csv

Training xgb...
Encoding labels for XGBoost...
Performing hyperparameter search...
✓ Best parameters: {'classifi

## 7. Compare Models and Select Best

In [17]:
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

# Create comparison DataFrame
comparison_data = []
for model_name, result in results.items():
    cv_df = result['cv_results']
    comparison_data.append({
        'Model': model_name,
        'Accuracy': f"{cv_df['accuracy'].mean():.4f} ± {cv_df['accuracy'].std():.4f}",
        'F1-Macro': f"{cv_df['f1_macro'].mean():.4f} ± {cv_df['f1_macro'].std():.4f}",
        'Precision': f"{cv_df['precision_macro'].mean():.4f} ± {cv_df['precision_macro'].std():.4f}",
        'Recall': f"{cv_df['recall_macro'].mean():.4f} ± {cv_df['recall_macro'].std():.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n", comparison_df.to_string(index=False))

# Select best model based on f1_macro
best_model_name = max(results.items(), key=lambda x: x[1]['mean_f1_macro'])[0]
best_pipeline = results[best_model_name]['pipeline']
best_f1 = results[best_model_name]['mean_f1_macro']
best_label_encoder = results[best_model_name]['label_encoder']

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_name}")
print(f"F1-Macro Score: {best_f1:.4f}")
print(f"{'='*60}")


MODEL COMPARISON SUMMARY

         Model        Accuracy        F1-Macro       Precision          Recall
random_forest 0.6282 ± 0.1018 0.6591 ± 0.0742 0.6892 ± 0.0839 0.6566 ± 0.0772
          xgb 0.5977 ± 0.0872 0.6360 ± 0.0637 0.6569 ± 0.0837 0.6290 ± 0.0563
           nb 0.3561 ± 0.0674 0.3419 ± 0.0891 0.3586 ± 0.1239 0.3586 ± 0.0774

BEST MODEL: random_forest
F1-Macro Score: 0.6591


## 8. Generate Labels Mapping

In [18]:
create_labels_mapping(y_train)


Labels mapping:
   label_index label_name  support
0            0          C       44
1            1         M1       39
2            2         M2       31
3            3         MM       18

✓ Labels mapping saved to labels_mapping.csv


## 9. Generate Detailed Reports for Best Model

In [19]:
generate_best_model_reports(best_pipeline, best_model_name, X_train, y_train, cv, best_label_encoder)


Generating reports for best model: random_forest
Generating cross-validated predictions...

Creating confusion matrix...
Confusion Matrix:
     C  M1  M2  MM
C   31  10   3   0
M1  19  17   3   0
M2   1   9  20   1
MM   0   0   3  15
✓ Confusion matrix saved to confusion_matrix.csv

Creating classification report...
Classification Report:
              precision    recall  f1-score     support
C              0.607843  0.704545  0.652632   44.000000
M1             0.472222  0.435897  0.453333   39.000000
M2             0.689655  0.645161  0.666667   31.000000
MM             0.937500  0.833333  0.882353   18.000000
accuracy       0.628788  0.628788  0.628788    0.628788
macro avg      0.676805  0.654734  0.663746  132.000000
weighted avg   0.631940  0.628788  0.628370  132.000000
✓ Classification report saved to classification_report.csv

Checking for feature importances...
Extracting feature importances...

Top 20 Feature Importances:
                         feature  importance
12    

## 10. Summary

In [20]:
print("\n" + "="*60)
print("EXECUTION COMPLETE")
print("="*60)
print("\nGenerated files:")
print("  - labels_mapping.csv")
for model_name in results.keys():
    print(f"  - cv_results_{model_name}.csv")
print("  - confusion_matrix.csv (best model)")
print("  - classification_report.csv (best model)")
if hasattr(best_pipeline.named_steps['classifier'], 'feature_importances_'):
    print("  - feature_importances.csv (best model)")
print("\n✓ All tasks completed successfully!")


EXECUTION COMPLETE

Generated files:
  - labels_mapping.csv
  - cv_results_random_forest.csv
  - cv_results_xgb.csv
  - cv_results_nb.csv
  - confusion_matrix.csv (best model)
  - classification_report.csv (best model)
  - feature_importances.csv (best model)

✓ All tasks completed successfully!


## 11. Comparación de Estrategias Jerárquicas

En esta sección compararemos tres enfoques diferentes para la clasificación jerárquica de miopía:

### Estrategias a evaluar:

1. **Top-Down (Descendente)**: 
   - Predice directamente DCombo (4 clases: C, M1, M2, MM)
   - Mapea jerárquicamente a M, MM y Combo
   - Ventajas: Simple, coherente, un solo modelo
   - Desventajas: Problema más complejo (4 clases)

2. **Bottom-Up (Ascendente/Cascada)**:
   - Nivel 1: Predice M (NO/SI)
   - Nivel 2: Si M=SI → Predice MM (NO/SI)
   - Nivel 3: Si M=SI y MM=NO → Predice M1 vs M2
   - Ventajas: Divide el problema, modelos especializados
   - Desventajas: Errores acumulativos en cascada

3. **Hybrid (Híbrido)**:
   - Combina Top-Down (principal) y Bottom-Up (validación)
   - Si hay conflicto y baja confianza → usa Bottom-Up
   - Ventajas: Más robusto, valida predicciones
   - Desventajas: Mayor complejidad computacional

### 11.1 Preparación de Datos para Análisis Jerárquico

In [21]:
# Preparar datos completos para las comparaciones
y_dict_full = {
    'M': Y_train_df['M'].values,
    'MM': Y_train_df['MM'].values,
    'Combo': Y_train_df['Combo'].values,
    'DCombo': Y_train_df['DCombo'].values
}

print("="*80)
print("DATOS PREPARADOS PARA COMPARACIÓN DE ESTRATEGIAS")
print("="*80)
print(f"\nMuestras totales: {len(y_dict_full['M'])}")
print(f"Columnas jerárquicas: {list(y_dict_full.keys())}")
print(f"\nDistribución de clases por columna:")
for col in ['DCombo', 'Combo', 'M', 'MM']:
    print(f"\n{col}:")
    print(pd.Series(y_dict_full[col]).value_counts().sort_index())

DATOS PREPARADOS PARA COMPARACIÓN DE ESTRATEGIAS

Muestras totales: 132
Columnas jerárquicas: ['M', 'MM', 'Combo', 'DCombo']

Distribución de clases por columna:

DCombo:
C     44
M1    39
M2    31
MM    18
Name: count, dtype: int64

Combo:
C     44
M     70
MM    18
Name: count, dtype: int64

M:
NO    44
SI    88
Name: count, dtype: int64

MM:
NO    122
SI     10
Name: count, dtype: int64


In [22]:
# Función auxiliar para mapear DCombo a otras columnas (usada en Top-Down)
def dcombo_to_hierarchical(dcombo_values):
    """
    Mapea valores de DCombo a las columnas M, MM y Combo.
    
    Reglas:
    - C → M=NO, MM=NO, Combo=C
    - M1 → M=SI, MM=NO, Combo=M
    - M2 → M=SI, MM=NO, Combo=M
    - MM → M=SI, MM=SI, Combo=MM
    """
    dcombo_array = np.array(dcombo_values)
    M = np.where(dcombo_array == 'C', 'NO', 'SI')
    MM = np.where(dcombo_array == 'MM', 'SI', 'NO')
    Combo = np.where(dcombo_array == 'C', 'C',
                    np.where(dcombo_array == 'MM', 'MM', 'M'))
    return M, MM, Combo

# Función para evaluar predicciones jerárquicas
def evaluate_hierarchical_predictions(y_true_dict, y_pred_dict, strategy_name=""):
    """
    Evalúa predicciones para todas las columnas jerárquicas.
    
    Parameters:
    -----------
    y_true_dict : dict
        Valores reales: {'M': array, 'MM': array, 'Combo': array, 'DCombo': array}
    y_pred_dict : dict
        Predicciones: {'M': array, 'MM': array, 'Combo': array, 'DCombo': array}
    strategy_name : str
        Nombre de la estrategia para el reporte
    
    Returns:
    --------
    results : dict
        Métricas para cada columna
    """
    print(f"\n{'='*80}")
    print(f"EVALUACIÓN: {strategy_name}")
    print(f"{'='*80}")
    
    results = {}
    
    for col in ['DCombo', 'Combo', 'M', 'MM']:
        y_true = y_true_dict[col]
        y_pred = y_pred_dict[col]
        
        acc = accuracy_score(y_true, y_pred)
        f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
        
        results[col] = {
            'accuracy': acc,
            'f1_macro': f1_macro
        }
        
        print(f"\n{col}:")
        print(f"  Accuracy:  {acc:.4f}")
        print(f"  F1-Macro:  {f1_macro:.4f}")
        print(f"\nClassification Report:")
        print(classification_report(y_true, y_pred, zero_division=0))
    
    return results

print("✓ Funciones auxiliares definidas")

✓ Funciones auxiliares definidas


### 11.2 Estrategia 1: TOP-DOWN (Descendente)

In [23]:
print("\n" + "="*80)
print("ESTRATEGIA 1: TOP-DOWN (Descendente)")
print("="*80)
print("\nDescripción:")
print("  - Predice DCombo directamente (4 clases: C, M1, M2, MM)")
print("  - Mapea jerárquicamente DCombo → Combo, M, MM")
print("  - Usa el mejor modelo ya entrenado (secciones anteriores)")

# Usar el mejor pipeline ya entrenado para DCombo
y_train_dcombo = y_dict_full['DCombo']

# Codificar si es necesario (para compatibilidad con el modelo)
if best_label_encoder is not None:
    y_train_dcombo_encoded = best_label_encoder.transform(y_train_dcombo)
else:
    y_train_dcombo_encoded = y_train_dcombo

# Obtener predicciones con cross-validation (predicciones "out-of-fold")
print("\nGenerando predicciones cross-validated (out-of-fold)...")
y_pred_dcombo = cross_val_predict(
    best_pipeline, 
    X_train, 
    y_train_dcombo_encoded,
    cv=cv, 
    method='predict'
)

# Decodificar si es necesario
if best_label_encoder is not None:
    y_pred_dcombo = best_label_encoder.inverse_transform(y_pred_dcombo)

# Mapear DCombo a otras columnas
M_pred, MM_pred, Combo_pred = dcombo_to_hierarchical(y_pred_dcombo)

y_pred_topdown = {
    'DCombo': y_pred_dcombo,
    'Combo': Combo_pred,
    'M': M_pred,
    'MM': MM_pred
}

print(f"\n✓ Predicciones Top-Down generadas")
print(f"  Total predicciones: {len(y_pred_dcombo)}")
print(f"  Distribución predicha DCombo: {pd.Series(y_pred_dcombo).value_counts().to_dict()}")


ESTRATEGIA 1: TOP-DOWN (Descendente)

Descripción:
  - Predice DCombo directamente (4 clases: C, M1, M2, MM)
  - Mapea jerárquicamente DCombo → Combo, M, MM
  - Usa el mejor modelo ya entrenado (secciones anteriores)

Generando predicciones cross-validated (out-of-fold)...

✓ Predicciones Top-Down generadas
  Total predicciones: 132
  Distribución predicha DCombo: {'C': 51, 'M1': 36, 'M2': 29, 'MM': 16}

✓ Predicciones Top-Down generadas
  Total predicciones: 132
  Distribución predicha DCombo: {'C': 51, 'M1': 36, 'M2': 29, 'MM': 16}


In [24]:
# Evaluar estrategia Top-Down
results_topdown = evaluate_hierarchical_predictions(
    y_dict_full, 
    y_pred_topdown, 
    strategy_name="TOP-DOWN"
)


EVALUACIÓN: TOP-DOWN

DCombo:
  Accuracy:  0.6288
  F1-Macro:  0.6637

Classification Report:
              precision    recall  f1-score   support

           C       0.61      0.70      0.65        44
          M1       0.47      0.44      0.45        39
          M2       0.69      0.65      0.67        31
          MM       0.94      0.83      0.88        18

    accuracy                           0.63       132
   macro avg       0.68      0.65      0.66       132
weighted avg       0.63      0.63      0.63       132


Combo:
  Accuracy:  0.7197
  F1-Macro:  0.7536

Classification Report:
              precision    recall  f1-score   support

           C       0.61      0.70      0.65        44
           M       0.75      0.70      0.73        70
          MM       0.94      0.83      0.88        18

    accuracy                           0.72       132
   macro avg       0.77      0.75      0.75       132
weighted avg       0.73      0.72      0.72       132


M:
  Accuracy:  

### 11.3 Estrategia 2: BOTTOM-UP (Ascendente/Cascada)

In [25]:
print("\n" + "="*80)
print("ESTRATEGIA 2: BOTTOM-UP (Ascendente/Cascada)")
print("="*80)
print("\nDescripción:")
print("  - Nivel 1: Predice M (NO/SI)")
print("  - Nivel 2: Si M=SI → Predice MM (NO/SI)")
print("  - Nivel 3: Si M=SI y MM=NO → Predice M1 vs M2")
print("  - Construye DCombo y Combo a partir de las decisiones en cascada")

# Preparar targets
y_m = y_dict_full['M']
y_mm = y_dict_full['MM']
y_dcombo = y_dict_full['DCombo']

# Nivel 1: Entrenar modelo M (NO/SI)
print("\n" + "-"*80)
print("NIVEL 1: Clasificador M (NO/SI)")
print("-"*80)
# Nota: X_train ya fue limpiado en sección 4, solo obtenemos preprocessor
preprocessor_m, _ = build_preprocessor(X_train, nan_threshold=0.20)
pipeline_m = Pipeline([
    ('preprocessor', preprocessor_m),
    ('classifier', RandomForestClassifier(
        n_estimators=150, 
        max_depth=10, 
        random_state=RANDOM_STATE, 
        n_jobs=-1
    ))
])
pipeline_m.fit(X_train, y_m)
print("✓ Modelo M entrenado")

# Nivel 2: Entrenar modelo MM solo con casos M=SI
print("\n" + "-"*80)
print("NIVEL 2: Clasificador MM (NO/SI) - Solo casos con M=SI")
print("-"*80)
mask_m_si = (y_m == 'SI')
X_m_si = X_train.loc[mask_m_si]
y_mm_si = y_mm[mask_m_si]
print(f"Muestras para entrenar MM: {len(X_m_si)} (de {len(X_train)} totales)")

# Nota: X_m_si es un subset, no necesita limpieza adicional
preprocessor_mm, _ = build_preprocessor(X_m_si, nan_threshold=0.20)
pipeline_mm = Pipeline([
    ('preprocessor', preprocessor_mm),
    ('classifier', RandomForestClassifier(
        n_estimators=150, 
        max_depth=10, 
        random_state=RANDOM_STATE, 
        n_jobs=-1
    ))
])
pipeline_mm.fit(X_m_si, y_mm_si)
print("✓ Modelo MM entrenado")

# Nivel 3: Entrenar modelo M1 vs M2 solo con casos M=SI y MM=NO
print("\n" + "-"*80)
print("NIVEL 3: Clasificador M1 vs M2 - Solo casos con M=SI y MM=NO")
print("-"*80)
# CORRECCIÓN: Filtrar por DCombo en lugar de MM para obtener solo M1 y M2
mask_m1m2 = ((y_dcombo == 'M1') | (y_dcombo == 'M2'))
X_m1m2 = X_train.loc[mask_m1m2]
y_dcombo_m1m2 = y_dcombo[mask_m1m2]
print(f"Muestras para entrenar M1vsM2: {len(X_m1m2)} (de {len(X_train)} totales)")
print(f"  M1: {(y_dcombo_m1m2 == 'M1').sum()}, M2: {(y_dcombo_m1m2 == 'M2').sum()}")

# Nota: X_m1m2 es un subset, no necesita limpieza adicional
preprocessor_m1m2, _ = build_preprocessor(X_m1m2, nan_threshold=0.20)
pipeline_m1m2 = Pipeline([
    ('preprocessor', preprocessor_m1m2),
    ('classifier', RandomForestClassifier(
        n_estimators=150, 
        max_depth=10, 
        random_state=RANDOM_STATE, 
        n_jobs=-1
    ))
])
pipeline_m1m2.fit(X_m1m2, y_dcombo_m1m2)
print("✓ Modelo M1 vs M2 entrenado")
print(f"  Clases del modelo: {pipeline_m1m2.named_steps['classifier'].classes_}")

print("\n" + "="*80)
print("✓ Entrenamiento en cascada completado")
print("="*80)


ESTRATEGIA 2: BOTTOM-UP (Ascendente/Cascada)

Descripción:
  - Nivel 1: Predice M (NO/SI)
  - Nivel 2: Si M=SI → Predice MM (NO/SI)
  - Nivel 3: Si M=SI y MM=NO → Predice M1 vs M2
  - Construye DCombo y Combo a partir de las decisiones en cascada

--------------------------------------------------------------------------------
NIVEL 1: Clasificador M (NO/SI)
--------------------------------------------------------------------------------
  Features finales: 46 columnas
    - Numéricas: 32
    - Categóricas: 14
✓ Modelo M entrenado

--------------------------------------------------------------------------------
NIVEL 2: Clasificador MM (NO/SI) - Solo casos con M=SI
--------------------------------------------------------------------------------
Muestras para entrenar MM: 88 (de 132 totales)
  Features finales: 46 columnas
    - Numéricas: 32
    - Categóricas: 14
✓ Modelo MM entrenado

--------------------------------------------------------------------------------
NIVEL 3: Clasificad

In [26]:
# Generar predicciones en cascada con CROSS-VALIDATION (out-of-fold)
print("\nGenerando predicciones Bottom-Up con cross-validation (out-of-fold)...")
n_samples = len(X_train)

# Nivel 1: Predecir M con CV
m_pred = cross_val_predict(pipeline_m, X_train, y_m, cv=cv)
print(f"  Nivel 1 (M): {(m_pred == 'SI').sum()} predicciones SI, {(m_pred == 'NO').sum()} predicciones NO")

# Inicializar arrays con dtype object para permitir cadenas de cualquier longitud
mm_pred = np.array(['NO'] * n_samples, dtype=object)
dcombo_pred = np.array(['C'] * n_samples, dtype=object)  # dtype=object para permitir 'M1', 'M2', 'MM'

# Nivel 2: Predecir MM solo donde M=SI (usando CV)
mask_m_si_pred = (m_pred == 'SI')
if mask_m_si_pred.any():
    X_m_si_pred = X_train.loc[mask_m_si_pred]
    y_mm_si_pred = y_mm[mask_m_si_pred]
    
    # Usar CV solo sobre el subset donde M=SI
    mm_pred_subset = cross_val_predict(pipeline_mm, X_m_si_pred, y_mm_si_pred, cv=cv)
    mm_pred[mask_m_si_pred] = mm_pred_subset
    print(f"  Nivel 2 (MM): {(mm_pred == 'SI').sum()} predicciones SI (de {mask_m_si_pred.sum()} casos M=SI)")
    
    # Casos donde M=SI y MM=SI → DCombo = MM
    mask_mm_si = mask_m_si_pred & (mm_pred == 'SI')
    dcombo_pred[mask_mm_si] = 'MM'
    
    # Nivel 3: Casos donde M=SI y MM=NO → Predecir M1 vs M2 (usando CV)
    mask_m1m2_pred = mask_m_si_pred & (mm_pred == 'NO')
    if mask_m1m2_pred.any():
        X_m1m2_pred = X_train.loc[mask_m1m2_pred]
        y_m1m2_pred = y_dcombo[mask_m1m2_pred]
        
        # Usar CV solo sobre el subset donde M=SI y MM=NO
        m1m2_pred_subset = cross_val_predict(pipeline_m1m2, X_m1m2_pred, y_m1m2_pred, cv=cv)
        dcombo_pred[mask_m1m2_pred] = m1m2_pred_subset
        
        # Contar M1 y M2
        m1_count = (dcombo_pred == 'M1').sum()
        m2_count = (dcombo_pred == 'M2').sum()
        print(f"  Nivel 3 (M1/M2): {m1_count} predicciones M1, {m2_count} predicciones M2")

# Derivar Combo de DCombo
combo_pred = np.where(dcombo_pred == 'C', 'C',
                     np.where(dcombo_pred == 'MM', 'MM', 'M'))

# Crear diccionario de predicciones
y_pred_bottomup = {
    'DCombo': dcombo_pred,
    'Combo': combo_pred,
    'M': m_pred,
    'MM': mm_pred
}

print(f"\n✓ Predicciones Bottom-Up generadas con cross-validation")
print(f"  Total predicciones: {len(dcombo_pred)}")
print(f"  Distribución predicha DCombo: {pd.Series(dcombo_pred).value_counts().to_dict()}")


Generando predicciones Bottom-Up con cross-validation (out-of-fold)...
  Nivel 1 (M): 99 predicciones SI, 33 predicciones NO
  Nivel 1 (M): 99 predicciones SI, 33 predicciones NO
  Nivel 2 (MM): 0 predicciones SI (de 99 casos M=SI)
  Nivel 2 (MM): 0 predicciones SI (de 99 casos M=SI)
  Nivel 3 (M1/M2): 41 predicciones M1, 33 predicciones M2

✓ Predicciones Bottom-Up generadas con cross-validation
  Total predicciones: 132
  Distribución predicha DCombo: {'M1': 41, 'C': 39, 'M2': 33, 'MM': 19}
  Nivel 3 (M1/M2): 41 predicciones M1, 33 predicciones M2

✓ Predicciones Bottom-Up generadas con cross-validation
  Total predicciones: 132
  Distribución predicha DCombo: {'M1': 41, 'C': 39, 'M2': 33, 'MM': 19}


In [27]:
# Evaluar estrategia Bottom-Up
results_bottomup = evaluate_hierarchical_predictions(
    y_dict_full, 
    y_pred_bottomup, 
    strategy_name="BOTTOM-UP (Cascada)"
)


EVALUACIÓN: BOTTOM-UP (Cascada)

DCombo:
  Accuracy:  0.5379
  F1-Macro:  0.5683

Classification Report:
              precision    recall  f1-score   support

           C       0.72      0.64      0.67        44
          M1       0.34      0.36      0.35        39
          M2       0.42      0.45      0.44        31
          MM       0.79      0.83      0.81        18

    accuracy                           0.54       132
   macro avg       0.57      0.57      0.57       132
weighted avg       0.55      0.54      0.54       132


Combo:
  Accuracy:  0.7424
  F1-Macro:  0.7498

Classification Report:
              precision    recall  f1-score   support

           C       0.72      0.64      0.67        44
           M       0.74      0.79      0.76        70
          MM       0.79      0.83      0.81        18

    accuracy                           0.74       132
   macro avg       0.75      0.75      0.75       132
weighted avg       0.74      0.74      0.74       132


M:
  

### 11.4 Estrategia 3: HYBRID (Híbrido con Validación)

In [28]:
print("\n" + "="*80)
print("ESTRATEGIA 3: HYBRID (Híbrido con Validación)")
print("="*80)
print("\nDescripción:")
print("  - Modelo principal: Top-Down (DCombo)")
print("  - Modelo de validación: Bottom-Up (M, MM)")
print("  - Regla: Si hay conflicto en M o MM Y confianza baja → usar Bottom-Up")
print("  - Umbral de confianza: 0.70 (70%)")

# Parámetro de confianza
CONFIDENCE_THRESHOLD = 0.70

# Obtener probabilidades del modelo Top-Down
print("\nCalculando probabilidades Top-Down...")
if best_label_encoder is not None:
    proba_topdown = best_pipeline.predict_proba(X_train)
else:
    proba_topdown = best_pipeline.predict_proba(X_train)

# Confianza = probabilidad máxima para cada predicción
confidence_topdown = proba_topdown.max(axis=1)

print(f"Confianza promedio Top-Down: {confidence_topdown.mean():.4f}")
print(f"Confianza mínima: {confidence_topdown.min():.4f}")
print(f"Confianza máxima: {confidence_topdown.max():.4f}")
print(f"Casos con confianza < {CONFIDENCE_THRESHOLD}: {(confidence_topdown < CONFIDENCE_THRESHOLD).sum()}")

# Inicializar con predicciones Top-Down
y_pred_hybrid = {
    'DCombo': y_pred_topdown['DCombo'].copy(),
    'Combo': y_pred_topdown['Combo'].copy(),
    'M': y_pred_topdown['M'].copy(),
    'MM': y_pred_topdown['MM'].copy()
}

# Validación cruzada: detectar conflictos y resolver con Bottom-Up
print("\n" + "-"*80)
print("VALIDACIÓN CRUZADA: Detectando y resolviendo conflictos")
print("-"*80)

conflicts_resolved = 0
n_samples = len(y_pred_topdown['DCombo'])

for i in range(n_samples):
    # Verificar conflictos en M o MM
    conflict_m = (y_pred_topdown['M'][i] != y_pred_bottomup['M'][i])
    conflict_mm = (y_pred_topdown['MM'][i] != y_pred_bottomup['MM'][i])
    
    # Si hay conflicto Y confianza baja → usar Bottom-Up
    if (conflict_m or conflict_mm) and confidence_topdown[i] < CONFIDENCE_THRESHOLD:
        conflicts_resolved += 1
        y_pred_hybrid['M'][i] = y_pred_bottomup['M'][i]
        y_pred_hybrid['MM'][i] = y_pred_bottomup['MM'][i]
        y_pred_hybrid['DCombo'][i] = y_pred_bottomup['DCombo'][i]
        y_pred_hybrid['Combo'][i] = y_pred_bottomup['Combo'][i]

print(f"\n✓ Conflictos detectados y resueltos: {conflicts_resolved}/{n_samples}")
print(f"  Porcentaje de casos resueltos con Bottom-Up: {conflicts_resolved/n_samples*100:.2f}%")
print(f"  Porcentaje de casos con Top-Down: {(n_samples-conflicts_resolved)/n_samples*100:.2f}%")
print(f"\nDistribución predicha DCombo (Hybrid): {pd.Series(y_pred_hybrid['DCombo']).value_counts().to_dict()}")


ESTRATEGIA 3: HYBRID (Híbrido con Validación)

Descripción:
  - Modelo principal: Top-Down (DCombo)
  - Modelo de validación: Bottom-Up (M, MM)
  - Regla: Si hay conflicto en M o MM Y confianza baja → usar Bottom-Up
  - Umbral de confianza: 0.70 (70%)

Calculando probabilidades Top-Down...
Confianza promedio Top-Down: 0.7638
Confianza mínima: 0.4332
Confianza máxima: 1.0000
Casos con confianza < 0.7: 48

--------------------------------------------------------------------------------
VALIDACIÓN CRUZADA: Detectando y resolviendo conflictos
--------------------------------------------------------------------------------

✓ Conflictos detectados y resueltos: 11/132
  Porcentaje de casos resueltos con Bottom-Up: 8.33%
  Porcentaje de casos con Top-Down: 91.67%

Distribución predicha DCombo (Hybrid): {'C': 42, 'M1': 42, 'M2': 32, 'MM': 16}


In [29]:
# Evaluar estrategia Hybrid
results_hybrid = evaluate_hierarchical_predictions(
    y_dict_full, 
    y_pred_hybrid, 
    strategy_name="HYBRID (Top-Down + Bottom-Up con validación)"
)


EVALUACIÓN: HYBRID (Top-Down + Bottom-Up con validación)

DCombo:
  Accuracy:  0.6667
  F1-Macro:  0.6975

Classification Report:
              precision    recall  f1-score   support

           C       0.71      0.68      0.70        44
          M1       0.52      0.56      0.54        39
          M2       0.66      0.68      0.67        31
          MM       0.94      0.83      0.88        18

    accuracy                           0.67       132
   macro avg       0.71      0.69      0.70       132
weighted avg       0.67      0.67      0.67       132


Combo:
  Accuracy:  0.7727
  F1-Macro:  0.7906

Classification Report:
              precision    recall  f1-score   support

           C       0.71      0.68      0.70        44
           M       0.77      0.81      0.79        70
          MM       0.94      0.83      0.88        18

    accuracy                           0.77       132
   macro avg       0.81      0.78      0.79       132
weighted avg       0.77      0.77   

### 11.5 Comparación Final de Estrategias

In [30]:
print("\n" + "="*80)
print("COMPARACIÓN FINAL DE LAS 3 ESTRATEGIAS")
print("="*80)

# Crear tabla comparativa
comparison_data = []

for strategy_name, results in [
    ('Top-Down', results_topdown), 
    ('Bottom-Up', results_bottomup),
    ('Hybrid', results_hybrid)
]:
    for col in ['DCombo', 'Combo', 'M', 'MM']:
        comparison_data.append({
            'Estrategia': strategy_name,
            'Columna': col,
            'Accuracy': results[col]['accuracy'],
            'F1-Macro': results[col]['f1_macro']
        })

comparison_df = pd.DataFrame(comparison_data)

# Mostrar tabla pivoteada
print("\nTabla Comparativa (Accuracy):")
print(comparison_df.pivot_table(
    index='Columna',
    columns='Estrategia',
    values='Accuracy'
).round(4))

print("\nTabla Comparativa (F1-Macro):")
print(comparison_df.pivot_table(
    index='Columna',
    columns='Estrategia',
    values='F1-Macro'
).round(4))

# Guardar comparación
comparison_df.to_csv('strategy_comparison.csv', index=False)
print("\n✓ Comparación guardada en 'strategy_comparison.csv'")


COMPARACIÓN FINAL DE LAS 3 ESTRATEGIAS

Tabla Comparativa (Accuracy):
Estrategia  Bottom-Up  Hybrid  Top-Down
Columna                                
Combo          0.7424  0.7727    0.7197
DCombo         0.5379  0.6667    0.6288
M              0.7955  0.8106    0.7500
MM             0.9242  0.9167    0.9091

Tabla Comparativa (F1-Macro):
Estrategia  Bottom-Up  Hybrid  Top-Down
Columna                                
Combo          0.7498  0.7906    0.7536
DCombo         0.5683  0.6975    0.6637
M              0.7525  0.7831    0.7287
MM             0.4803  0.7570    0.7440

✓ Comparación guardada en 'strategy_comparison.csv'


In [31]:
# Calcular promedios por estrategia
print("\n" + "="*80)
print("ANÁLISIS DE RENDIMIENTO PROMEDIO")
print("="*80)

for strategy in ['Top-Down', 'Bottom-Up', 'Hybrid']:
    strategy_data = comparison_df[comparison_df['Estrategia'] == strategy]
    avg_acc = strategy_data['Accuracy'].mean()
    avg_f1 = strategy_data['F1-Macro'].mean()
    
    print(f"\n{strategy}:")
    print(f"  Accuracy promedio:  {avg_acc:.4f}")
    print(f"  F1-Macro promedio:  {avg_f1:.4f}")

# Determinar mejor estrategia
best_strategy_acc = comparison_df.groupby('Estrategia')['Accuracy'].mean().idxmax()
best_acc = comparison_df.groupby('Estrategia')['Accuracy'].mean().max()

best_strategy_f1 = comparison_df.groupby('Estrategia')['F1-Macro'].mean().idxmax()
best_f1 = comparison_df.groupby('Estrategia')['F1-Macro'].mean().max()

print("\n" + "="*80)
print("MEJORES ESTRATEGIAS")
print("="*80)
print(f"\nMejor por Accuracy:  {best_strategy_acc} ({best_acc:.4f})")
print(f"Mejor por F1-Macro:  {best_strategy_f1} ({best_f1:.4f})")


ANÁLISIS DE RENDIMIENTO PROMEDIO

Top-Down:
  Accuracy promedio:  0.7519
  F1-Macro promedio:  0.7225

Bottom-Up:
  Accuracy promedio:  0.7500
  F1-Macro promedio:  0.6377

Hybrid:
  Accuracy promedio:  0.7917
  F1-Macro promedio:  0.7570

MEJORES ESTRATEGIAS

Mejor por Accuracy:  Hybrid (0.7917)
Mejor por F1-Macro:  Hybrid (0.7570)


### 11.6 Conclusiones y Recomendaciones

In [32]:
print("\n" + "="*80)
print("CONCLUSIONES Y RECOMENDACIONES FINALES")
print("="*80)

# Calcular diferencias
avg_scores = comparison_df.groupby('Estrategia')[['Accuracy', 'F1-Macro']].mean()

print("\nRESUMEN DE RENDIMIENTO:")
print("-" * 80)
print(avg_scores.round(4))

# Análisis comparativo
topdown_acc = avg_scores.loc['Top-Down', 'Accuracy']
bottomup_acc = avg_scores.loc['Bottom-Up', 'Accuracy']
hybrid_acc = avg_scores.loc['Hybrid', 'Accuracy']

print("\n" + "="*80)
print("ANÁLISIS DETALLADO")
print("="*80)

# Comparar Top-Down vs Bottom-Up
diff_topdown_bottomup = (bottomup_acc - topdown_acc) * 100
print(f"\n1. Top-Down vs Bottom-Up:")
if bottomup_acc > topdown_acc:
    print(f"   ✓ Bottom-Up es MEJOR que Top-Down por {diff_topdown_bottomup:.2f}%")
    print(f"   → Bottom-Up divide el problema y especializa cada decisión")
elif topdown_acc > bottomup_acc:
    print(f"   ✓ Top-Down es MEJOR que Bottom-Up por {-diff_topdown_bottomup:.2f}%")
    print(f"   → Top-Down es más simple y evita errores acumulativos")
else:
    print(f"   ≈ Ambas estrategias tienen rendimiento similar")

# Comparar Hybrid vs las otras
diff_hybrid_topdown = (hybrid_acc - topdown_acc) * 100
diff_hybrid_bottomup = (hybrid_acc - bottomup_acc) * 100

print(f"\n2. Hybrid vs Top-Down:")
if hybrid_acc > topdown_acc:
    print(f"   ✓ Hybrid es MEJOR que Top-Down por {diff_hybrid_topdown:.2f}%")
elif topdown_acc > hybrid_acc:
    print(f"   ✗ Top-Down es mejor que Hybrid por {-diff_hybrid_topdown:.2f}%")
else:
    print(f"   ≈ Rendimiento similar")

print(f"\n3. Hybrid vs Bottom-Up:")
if hybrid_acc > bottomup_acc:
    print(f"   ✓ Hybrid es MEJOR que Bottom-Up por {diff_hybrid_bottomup:.2f}%")
elif bottomup_acc > hybrid_acc:
    print(f"   ✗ Bottom-Up es mejor que Hybrid por {-diff_hybrid_bottomup:.2f}%")
else:
    print(f"   ≈ Rendimiento similar")

# Recomendación final
print("\n" + "="*80)
print("RECOMENDACIÓN FINAL")
print("="*80)

best_overall = avg_scores['Accuracy'].idxmax()
best_score = avg_scores['Accuracy'].max()

print(f"\n🏆 ESTRATEGIA RECOMENDADA: {best_overall}")
print(f"   Accuracy promedio: {best_score:.4f}")

if best_overall == 'Top-Down':
    print("\n   VENTAJAS:")
    print("   ✓ Más simple de implementar y mantener")
    print("   ✓ Un solo modelo que entrenar")
    print("   ✓ Coherencia jerárquica garantizada por diseño")
    print("   ✓ Menor complejidad computacional")
    
elif best_overall == 'Bottom-Up':
    print("\n   VENTAJAS:")
    print("   ✓ Divide el problema en decisiones más simples")
    print("   ✓ Cada modelo se especializa en una decisión específica")
    print("   ✓ Mejor para datos desbalanceados")
    print("   ✓ Puede optimizarse cada nivel independientemente")
    
else:  # Hybrid
    print("\n   VENTAJAS:")
    print("   ✓ Combina lo mejor de ambos enfoques")
    print("   ✓ Validación cruzada entre estrategias")
    print("   ✓ Más robusto ante casos dudosos")
    print("   ✓ Usa Bottom-Up solo cuando hay incertidumbre")

print("\n" + "="*80)
print("PRÓXIMOS PASOS SUGERIDOS")
print("="*80)
print("""
1. Validar con conjunto de test independiente
2. Analizar casos específicos donde cada estrategia falla
3. Considerar ajustar hiperparámetros de modelos Bottom-Up
4. Evaluar con cross-validation completo (incluir fold en cascada)
5. Implementar la estrategia recomendada en producción
6. Monitorear rendimiento en datos reales
""")


CONCLUSIONES Y RECOMENDACIONES FINALES

RESUMEN DE RENDIMIENTO:
--------------------------------------------------------------------------------
            Accuracy  F1-Macro
Estrategia                    
Bottom-Up     0.7500    0.6377
Hybrid        0.7917    0.7570
Top-Down      0.7519    0.7225

ANÁLISIS DETALLADO

1. Top-Down vs Bottom-Up:
   ✓ Top-Down es MEJOR que Bottom-Up por 0.19%
   → Top-Down es más simple y evita errores acumulativos

2. Hybrid vs Top-Down:
   ✓ Hybrid es MEJOR que Top-Down por 3.98%

3. Hybrid vs Bottom-Up:
   ✓ Hybrid es MEJOR que Bottom-Up por 4.17%

RECOMENDACIÓN FINAL

🏆 ESTRATEGIA RECOMENDADA: Hybrid
   Accuracy promedio: 0.7917

   VENTAJAS:
   ✓ Combina lo mejor de ambos enfoques
   ✓ Validación cruzada entre estrategias
   ✓ Más robusto ante casos dudosos
   ✓ Usa Bottom-Up solo cuando hay incertidumbre

PRÓXIMOS PASOS SUGERIDOS

1. Validar con conjunto de test independiente
2. Analizar casos específicos donde cada estrategia falla
3. Considerar 